In [7]:
"""
opr_8_find_diff.ipynb
Compare summaries between old (non-conversational) and new (conversational) evaluation.
Focus on: wrong classifications and differently classified items.
"""
import pandas as pd
from pathlib import Path

# === PATHS ===
OLD_PATH = Path("df_text_multi_eval_by_victim.csv")
NEW_PATH = Path("results_large_models/df_text_by_report_conversation_evaluation.csv")

In [8]:
# === LOAD OLD (non-conversational) ===
df_old = pd.read_csv(OLD_PATH, encoding="utf-8")
print(f"OLD: {len(df_old)} rows from {OLD_PATH}")

# === LOAD NEW (conversational) ===
df_new_raw = pd.read_csv(NEW_PATH, encoding="utf-8")
print(f"NEW (raw): {len(df_new_raw)} rows from {NEW_PATH}")

# Filter: keep only the last report per victim
df_new = (
    df_new_raw.sort_values(by=["victim", "index"])
    .groupby("victim", as_index=False)
    .tail(1)
)
print(f"NEW (last per victim): {len(df_new)} rows")

OLD: 1633 rows from df_text_multi_eval_by_victim.csv
NEW (raw): 2229 rows from results_large_models\df_text_by_report_conversation_evaluation.csv
NEW (last per victim): 575 rows


In [9]:
SUMMARY_COL = "summary_all_context"

def filter_by_match_change(
    df_old: pd.DataFrame,
    df_new: pd.DataFrame,
    match_cols: list[str],
    mode: str = "improved",
    key_col: str = "victim",
    summary_col: str = SUMMARY_COL,
    old_threshold: float = 0.5,
    new_threshold: float = 0.8,
) -> pd.DataFrame:
    """
    Filter rows by match change between old and new.
    
    Parameters:
        df_old: Old dataframe
        df_new: New dataframe  
        match_cols: List of match column names
        mode: "improved" (old wrong, new right) or "regressed" (old right, new wrong)
        key_col: Column to join on
        summary_col: Summary column name to include
        old_threshold: Threshold for old mean_match
        new_threshold: Threshold for new mean_match
    
    Returns:
        Merged dataframe filtered by mode
        Includes: key_col, old_summary, new_summary, old_mean_match, new_mean_match
    """
    # Compute mean match for each row
    old_match_cols = [c for c in match_cols if c in df_old.columns]
    new_match_cols = [c for c in match_cols if c in df_new.columns]
    
    df_old_copy = df_old.copy()
    df_new_copy = df_new.copy()
    
    df_old_copy["mean_match"] = df_old_copy[old_match_cols].mean(axis=1)
    df_new_copy["mean_match"] = df_new_copy[new_match_cols].mean(axis=1)
    
    # Columns to keep from each dataframe
    old_cols = [key_col, summary_col, "mean_match"]
    new_cols = [key_col, summary_col, "mean_match"]
    
    # Merge on key column, include summary columns
    df_merged = pd.merge(
        df_old_copy[old_cols].rename(columns={summary_col: "old_summary", "mean_match": "old_mean_match"}),
        df_new_copy[new_cols].rename(columns={summary_col: "new_summary", "mean_match": "new_mean_match"}),
        on=key_col,
        how="inner"
    )
    
    # Filter based on mode
    if mode == "improved":
        # old wrong (< threshold), new right (>= threshold)
        mask = (df_merged["old_mean_match"] < old_threshold) & (df_merged["new_mean_match"] >= new_threshold)
    elif mode == "regressed":
        # old right (>= threshold), new wrong (< threshold)
        mask = (df_merged["old_mean_match"] >= old_threshold) & (df_merged["new_mean_match"] < new_threshold)
    else:
        raise ValueError(f"mode must be 'improved' or 'regressed', got '{mode}'")
    
    return df_merged[mask]

In [10]:
# Find match columns common to both dataframes
match_cols_old = [c for c in df_old.columns if c.endswith("_match")]
match_cols_new = [c for c in df_new.columns if c.endswith("_match")]
match_cols_common = list(set(match_cols_old) & set(match_cols_new))
print(f"Common match columns: {len(match_cols_common)}")

Common match columns: 15


In [11]:
# === FIND IMPROVEMENTS: OLD WRONG, NEW RIGHT ===
df_improved = filter_by_match_change(
    df_old=df_old,
    df_new=df_new,
    match_cols=match_cols_common,
    mode="improved",
    old_threshold=0.5,
    new_threshold=0.8
)
print(f"IMPROVED - OLD wrong (<0.5), NEW right (>=0.8): {len(df_improved)} rows")
df_improved

IMPROVED - OLD wrong (<0.5), NEW right (>=0.8): 141 rows


,victim,old_summary,old_mean_match,new_summary,new_mean_match
67,Coahuila_Jorge B B,La Policía Federal no encuentra ni a los suyos...,0.266667,NaN,0.800000
246,Jalisco_Daniel Eduardo C B,La desaparición de Daniel Eduardo Carpio Bocan...,0.400000,NaN,0.800000
302,Jalisco_Luis Fernando R M,"El 7 de julio de 2013, siete jóvenes y un adul...",0.400000,],0.866667
318,Jalisco_Miguel Ángel C O,"El 7 de julio de 2013, siete personas fueron s...",0.200000,NaN,0.866667
325,Jalisco_Oscar Alberto C M,El artículo describe la desaparición de Juan C...,0.333333,NaN,0.933333
...,...,...,...,...,...
1628,Veracruz_Xochitl Celeste C H,El caso de **Xóchitl Celeste Castañeda Hernánd...,0.266667,NaN,1.000000
1629,Veracruz_Yael Zuriel M J,El texto describe una situación crítica de **d...,0.266667,NaN,1.000000
1630,Veracruz_Yair D P,El texto aborda casos de **desapariciones forz...,0.133333,NaN,1.000000
1631,Veracruz_Yolatl Thalia B H,El texto aborda casos de **desapariciones forz...,0.133333,NaN,1.000000


In [12]:
# === SAVE TO CSV ===
OUTPUT_DIR = Path("temp_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Save without text columns (df_improved only has: victim, old_mean_match, new_mean_match)
output_path = OUTPUT_DIR / "old_wrong_new_right.csv"
df_improved.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved {len(df_improved)} rows to {output_path}")

Saved 141 rows to temp_results\old_wrong_new_right.csv


In [13]:
# === FIND REGRESSIONS: OLD RIGHT, NEW WRONG ===
df_regressed = filter_by_match_change(
    df_old=df_old,
    df_new=df_new,
    match_cols=match_cols_common,
    mode="regressed",
    old_threshold=0.8,
    new_threshold=0.5
)
print(f"REGRESSED - OLD right (>=0.8), NEW wrong (<0.5): {len(df_regressed)} rows")
df_regressed

REGRESSED - OLD right (>=0.8), NEW wrong (<0.5): 22 rows


,victim,old_summary,old_mean_match,new_summary,new_mean_match
438,Nuevo León_Ricardo R Gu,El neurólogo Ricardo Rangel Guerra fue secuest...,0.800000,NaN,0.466667
457,Veracruz_Ambar Nayeli S R,NaN,1.000000,La madre de la joven ausente jura que el gobie...,0.333333
461,Veracruz_Argenis Yosimar P B,NaN,1.000000,La desaparición de personas en México es un pr...,0.466667
471,Veracruz_Cristo Dassaiev B R,NaN,1.000000,La Procuradur\u00eda General de la Rep\u00fabl...,0.400000
477,Veracruz_Dorian Javier R Z,NaN,1.000000,Los colectivos de madres de desaparecidos loca...,0.400000
478,Veracruz_Eber Arturo C D,NaN,1.000000,Los desaparecidos son Jos\u00e9 Manuel Cruz P\...,0.400000
489,Veracruz_Filiberto A M,NaN,1.000000,"Consta en esa denuncia, que al menos dos patru...",0.333333
524,Veracruz_Karla Nayelli S H,NaN,1.000000,Desapariciones se ocultaron por órdenes de man...,0.466667
526,Veracruz_Levi R R,NaN,1.000000,Los marinos participaron en el homicidio y des...,0.400000
532,Veracruz_Maira Irazema R G,NaN,1.000000,"La joven Maira Irazema Rodríguez González, de ...",0.400000


In [18]:
# === FIND ROWS THAT REGRESSED ON SPECIFIC COLUMNS ===
# These columns showed worse performance with larger model
REGRESSED_COLS = ["desenlace_match", "captura_metodo_match", "soc_civil_match"]

# Merge old and new with all three columns
df_merged = pd.merge(
    df_old[["victim", SUMMARY_COL] + REGRESSED_COLS],
    df_new[["victim", SUMMARY_COL] + REGRESSED_COLS],
    on="victim",
    suffixes=("_old", "_new")
)

# Exclude Veracruz
df_merged = df_merged[~df_merged["victim"].str.startswith("Veracruz")]

# Filter: ALL three columns regressed (old=1, new=0)
mask = True
for col in REGRESSED_COLS:
    mask = mask & ((df_merged[f"{col}_old"] == 1) & (df_merged[f"{col}_new"] == 0))

df_regressed = df_merged[mask]
print(f"Rows where ALL of {REGRESSED_COLS} regressed: {len(df_regressed)}")
display(df_regressed)

Rows where ALL of ['desenlace_match', 'captura_metodo_match', 'soc_civil_match'] regressed: 94


,victim,summary_all_context_old,desenlace_match_old,captura_metodo_match_old,soc_civil_match_old,summary_all_context_new,desenlace_match_new,captura_metodo_match_new,soc_civil_match_new
4,Coahuila_Antonio H H,La Fiscalía de Personas Desaparecidas en Coahu...,1.0,1.0,1.0,NaN,0.0,0.0,0.0
6,Coahuila_Antonio de Jesus V E,"El 24 de enero de 2009, José Antonio Verástegu...",1.0,1.0,1.0,NaN,0.0,0.0,0.0
33,Coahuila_Erika Victoria P C,La joven Erika Victoria Prone Ceniceros desapa...,1.0,1.0,1.0,NaN,0.0,0.0,0.0
58,Coahuila_Javier G V,"Dos personas, José Angel Nicanor Reyes y Javie...",1.0,1.0,1.0,NaN,0.0,0.0,0.0
60,Coahuila_Jesus Antonio M C,La desaparición de Jesús Antonio Mena Contrera...,1.0,1.0,1.0,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
1474,Nuevo León_Kristian Karim F H,El texto aborda el caso de **Kristian Karim Fl...,1.0,1.0,1.0,NaN,0.0,0.0,0.0
1496,Nuevo León_Ricardo R Gu,"El neurólogo **Ricardo Rangel Guerra**, de **7...",1.0,1.0,1.0,NaN,0.0,0.0,0.0
1500,Nuevo León_Samantha M C,"El caso de **Samantha Briseida Moya Cantú**, u...",1.0,1.0,1.0,NaN,0.0,0.0,0.0
1504,Nuevo León_Tomas B G,El regidor perredista **Tomás Betancourt Gaitá...,1.0,1.0,1.0,NaN,0.0,0.0,0.0


In [15]:
# === SAVE REGRESSIONS TO CSV ===
output_path_regressed = OUTPUT_DIR / "old_right_new_wrong.csv"
df_regressed.to_csv(output_path_regressed, index=False, encoding="utf-8")
print(f"Saved {len(df_regressed)} rows to {output_path_regressed}")

Saved 22 rows to temp_results\old_right_new_wrong.csv
